# Limpieza y ingeniería de atributos
Este notebook va a ser para limpiar el dataframe.

Se recibe como entrada un dataframe (puede ser el de cualquier año o el consolidado total) y se van a procesar y limpiar las siguientes columnas:

- Género: Poner únicamente los géneros en F, M, No especificado
- Edad: Verificar que no haya edades negativas o que no cuadren (muy bajas o muy altas)
- Fechas: Verificar que estén en formato DD/MM/AAAA
- Eliminar nulos y duplicados

# Bibliotecas

In [2]:
%pip install polars

   ---------------------------------------- 0.0/783.6 kB ? eta -:--:--
   ---------------------------------------- 783.6/783.6 kB 8.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
   -- ------------------------------------- 2.4/41.3 MB 11.7 MB/s eta 0:00:04
   ---- ----------------------------------- 4.7/41.3 MB 11.3 MB/s eta 0:00:04
   ------ --------------------------------- 6.6/41.3 MB 10.9 MB/s eta 0:00:04
   -------- ------------------------------- 8.7/41.3 MB 10.6 MB/s eta 0:00:04
   ---------- ----------------------------- 10.7/41.3 MB 10.5 MB/s eta 0:00:03
   ------------ --------------------------- 12.8/41.3 MB 10.3 MB/s eta 0:00:03
   -------------- ------------------------- 14.9/41.3 MB 10.3 MB/s eta 0:00:03
   ---------------- ----------------------- 17.0/41.3 MB 10.3 MB/s eta 0:00:03
   ------------------ --------------------- 18.6/41.3 MB 10.1 MB/s eta 0:00:03
   ------------------- -------------------- 20.2/41.3 MB 9.8 MB/s eta 0:


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import polars as pl
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import re
from collections import Counter

# Obtenemos los datos de entrada

In [24]:
df = pl.read_parquet("DATA/consolidado_2017.parquet")
df

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
str,str,str,str,str,str,str,str,str,str
"""M""","""42""","""1570""","""33""","""01/01/2017""","""0:05:06""","""1""","""01/01/2017""","""0:16:49""","""2017/1.csv"""
"""M""","""36""","""7083""","""27""","""01/01/2017""","""0:10:13""","""123""","""01/01/2017""","""0:18:28""","""2017/1.csv"""
"""M""","""18""","""4093""","""157""","""01/01/2017""","""0:14:06""","""157""","""01/01/2017""","""1:05:31""","""2017/1.csv"""
"""M""","""20""","""7704""","""87""","""01/01/2017""","""0:14:35""","""43""","""01/01/2017""","""0:18:27""","""2017/1.csv"""
"""M""","""34""","""7176""","""47""","""01/01/2017""","""0:16:54""","""123""","""01/01/2017""","""0:26:08""","""2017/1.csv"""
…,…,…,…,…,…,…,…,…,…
"""M""","""25""","""10504""","""326""","""29/09/2017""","""23:58:50""","""148""","""30/09/2017""","""0:22:37""","""2017/9.csv"""
"""M""","""32""","""4367""","""329""","""29/09/2017""","""23:58:56""","""393""","""30/09/2017""","""0:16:30""","""2017/9.csv"""
"""M""","""38""","""9543""","""64""","""29/09/2017""","""23:59:34""","""74""","""30/09/2017""","""0:03:57""","""2017/9.csv"""


# Eliminamos nulos y duplicados

In [14]:
# Verificar cuántos nulos hay
df.null_count()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [4]:
# Rellenar con "No especificado" todos los nulos existentes
df = df.fill_null("No especificado")

In [5]:
# Verificamos que ya no haya nulos
df.null_count()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [6]:
# Ver cuántos duplicados hay
df.is_duplicated().sum()

0

In [7]:
#Para eliminar duplicados
df.unique()

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
str,str,str,str,str,str,str,str,str,str
"""M""","""26""","""6546""","""27""","""2016-01-22""","""15:24:36.250000""","""4""","""2016-01-22""","""15:35:58.157000""","""2016/1.csv"""
"""F""","""33""","""5186""","""89""","""2016-02-16""","""13:07:38.740000""","""101""","""2016-02-16""","""13:13:28""","""2016/2.csv"""
"""F""","""18""","""3476""","""130""","""2016-05-16""","""18:32:41.930000""","""141""","""2016-05-16""","""18:41:33.607000""","""2016/5.csv"""
"""F""","""56""","""5320""","""76""","""2016-01-12""","""10:28:12.173000""","""156""","""2016-01-12""","""10:35:25""","""2016/1.csv"""
"""M""","""35""","""2962""","""63""","""26/08/2016""","""23:40:02""","""289""","""26/08/2016""","""23:49:51""","""2016/8.csv"""
…,…,…,…,…,…,…,…,…,…
"""F""","""29""","""6072""","""182""","""2016-04-09""","""10:54:19.687000""","""80""","""2016-04-09""","""10:57:56.843000""","""2016/4.csv"""
"""M""","""29""","""2196""","""329""","""19/10/2016""","""19:12:28""","""325""","""19/10/2016""","""19:27:04""","""2016/10.csv"""
"""F""","""24""","""6162""","""158""","""2016-03-29""","""19:17:45.137000""","""135""","""2016-03-29""","""19:27:46""","""2016/3.csv"""


In [8]:
# Ver cuántos duplicados hay
df.is_duplicated().sum()

0

# Tratamiento a columna "Genero_Usuario"
Solo vamos a tomar en cuenta los registros que sean 'F', 'M' y 'No especificado'. Los registros que no sean 'F' o 'M', serán reemplazados por 'No especificado'.

Estos registros ya son conocidos, son los valores: 'O', '?' y 'nan'

In [15]:
valores_no_estandar = ['O', '?', 'nan'] 
df = df.with_columns(
    pl.col("Genero_Usuario").replace(valores_no_estandar, 'No especificado')
)

print("\nValores de 'Genero_Usuario' después de la limpieza:")
print(df['Genero_Usuario'].value_counts())


Valores de 'Genero_Usuario' después de la limpieza:
shape: (2, 2)
┌────────────────┬─────────┐
│ Genero_Usuario ┆ count   │
│ ---            ┆ ---     │
│ str            ┆ u32     │
╞════════════════╪═════════╡
│ M              ┆ 6810462 │
│ F              ┆ 2259081 │
└────────────────┴─────────┘


# Tratamiento a columnas de fechas

In [ ]:
#Revisar si podemos estandarizar los años con horas extrañas, sino, solo trabajamos desde 2017
df = df.with_columns(
    pl.col("Fecha_Retiro")
      .cast(pl.Utf8)
      .str.replace("-", "/")
      .str.strptime(pl.Date, format="%d/%m/%Y", strict=False),

    pl.col("Fecha_Arribo")
      .cast(pl.Utf8)
      .str.replace("-", "/")
      .str.strptime(pl.Date, format="%d/%m/%Y", strict=False)
)

print("2010 procesado correctamente")

2010 procesado correctamente


In [ ]:
#Revisar si podemos estandarizar los años con horas extrañas, sino, solo trabajamos desde 2017
def fix_time(col):
    return (
        pl.col(col)
        # Si falta microsegundos: agregar ".000000"
        .str.replace_all(r"^(\d{1,2}:\d{2}:\d{2})$", r"\1.000000")
        # Si la hora tiene un solo dígito al inicio: anteponer "0"
        .str.replace_all(r"^(\d:)", r"0\1")
        # Convertir a time
        .str.strptime(pl.Time, format="%H:%M:%S%.f", strict=False)
    )

df = df.with_columns(
    [
        fix_time("Hora_Retiro").alias("Hora_Retiro"),
        fix_time("Hora_Arribo").alias("Hora_Arribo"),
    ]
)

df

Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo,_source_file
str,str,str,str,date,time,str,date,time,str
"""M""","""51""","""5558""","""74""",null,00:01:59.413,"""80""",null,null,"""2016/1.csv"""
"""F""","""34""","""4760""","""123""",null,00:03:44.340,"""30""",null,00:11:07.463,"""2016/1.csv"""
"""M""","""29""","""1599""","""91""",null,00:04:02.280,"""29""",null,null,"""2016/1.csv"""
"""M""","""25""","""5840""","""138""",null,00:05:55.833,"""155""",null,null,"""2016/1.csv"""
"""M""","""24""","""1532""","""29""",null,00:08:14.827,"""11""",null,null,"""2016/1.csv"""
…,…,…,…,…,…,…,…,…,…
"""M""","""45""","""3834""","""88""",2016-09-29,null,"""260""",2016-09-30,null,"""2016/9.csv"""
"""F""","""24""","""7157""","""303""",2016-09-29,null,"""303""",2016-09-30,null,"""2016/9.csv"""
"""M""","""27""","""6817""","""27""",2016-09-29,null,"""1""",2016-09-30,null,"""2016/9.csv"""
